In [9]:
import polars as pl


df = pl.read_parquet(
        "datasets/dataset_merged_with_families.parquet",
    )

df


original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,cath_dominant,cath_all,cath_class,cath_arch,cath_topology,cath_homology
str,str,str,f64,bool,str,str,str,str,str,str,str
"""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""I27T""",-0.352598,false,"""megascale""","""G3DSA:1.20.900.10""","""G3DSA:1.10.238.10;G3DSA:1.20.9…","""G3DSA:1""","""20""","""900""","""10"""
"""SAGGSAEIMKKTDFDKVASEYTKIGTISTT…","""SAGGSAEIMKKTDFDKVASEYTLIGTISTT…","""K18L:D65P""",-0.157241,false,"""megascale""","""G3DSA:1.10.287.540""","""G3DSA:1.10.287.540""","""G3DSA:1""","""10""","""287""","""540"""
"""SAGGSEVTIKANLIFANGSTQTAEFKGTFE…","""SAGGSEVTIKANLIFANGSGQTAEFKGTFE…","""T15G""",-0.166278,false,"""megascale""","""G3DSA:1.10.10.10""","""G3DSA:1.10.10.10;G3DSA:1.10.15…","""G3DSA:1""","""10""","""10""","""10"""
"""SAGGSAVTTYKLVINGKTLKGETTTKAVDA…","""SAGGSAVTTYKRVINGKTLKGETTTKAVDA…","""L6R""",-0.580267,false,"""megascale""","""G3DSA:1.20.1270.60""","""G3DSA:1.20.1270.60;G3DSA:2.30.…","""G3DSA:1""","""20""","""1270""","""60"""
"""SAGGSAGGSAGGTTYKLILNGKTLKGETTT…","""SAGGSAGGSAGGTTYKHILNGKTLKGETTT…","""L5H:F30N""",-0.675053,false,"""megascale""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60"""
…,…,…,…,…,…,…,…,…,…,…,…
"""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""S906I""",-0.140346,false,"""lehner""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60"""
"""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""T906I""",-0.183833,false,"""lehner""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60"""
"""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""MSNVSEERRKRQQNIKEGLQFIQSPLSYPG…","""V906I""",-0.11009,false,"""lehner""","""G3DSA:1.10.10.60""","""G3DSA:1.10.10.60""","""G3DSA:1""","""10""","""10""","""60"""


In [30]:
import polars as pl
import random
import os

# --- NASTAVENÍ ---
# df = ... (Váš načtený DataFrame)

HOMOLOGY_COL = "cath_dominant"
SEED = 42
BASE_DIR = "datasets"
FILE_PREFIX = "dataset_homology_split_"
FULL_PREFIX = os.path.join(BASE_DIR, FILE_PREFIX)

# Přejmenování na standardní názvy (zachováme celé sekvence)
RENAME_MAP = {
    "original_seq_full": "wt_sequence",
    "mutated_seq_full": "mut_sequence",
    "target": "fitness",
    "mut_type": "mutation",
    "cath_dominant": "cluster_id"
}

os.makedirs(BASE_DIR, exist_ok=True)
print("Zahajuji deterministický split (zachování celých sekvencí)...")

# 1. Příprava rodin (Deterministická)
family_counts = (
    df.group_by(HOMOLOGY_COL)
    .len()
    .sort(HOMOLOGY_COL) # Nutné pro determinismus
    .to_dicts()
)

random.seed(SEED)
random.shuffle(family_counts)

# 2. Split Train vs Holdout (80/20 rodin)
total_rows = len(df)
holdout_goal = int(total_rows * 0.2)
train_fams, holdout_fams = [], []
curr_holdout = 0

for fam in family_counts:
    if curr_holdout < holdout_goal:
        holdout_fams.append(fam[HOMOLOGY_COL])
        curr_holdout += fam["len"]
    else:
        train_fams.append(fam[HOMOLOGY_COL])

# 3. Vytvoření Datasetů
train_df = df.filter(pl.col(HOMOLOGY_COL).is_in(train_fams))
holdout_df = df.filter(pl.col(HOMOLOGY_COL).is_in(holdout_fams))

# 4. Split Holdout na Val/Test (1:1)
# Seřadíme před mícháním pro determinismus
holdout_df = holdout_df.sort([HOMOLOGY_COL, "original_seq_full"])
holdout_df = holdout_df.sample(fraction=1.0, shuffle=True, seed=SEED)

split_point = len(holdout_df) // 2
val_df = holdout_df.slice(0, split_point)
test_df = holdout_df.slice(split_point, len(holdout_df) - split_point)

selected_column = ["wt_sequence", "mut_sequence", "mutation", "fitness", "cath_class", "cath_arch", "cath_topology", "cath_homology", "data_source"]

# 5. Uložení s přejmenováním
def save(d, name):
    valid_map = {k: v for k, v in RENAME_MAP.items() if k in d.columns}
    d.rename(valid_map).select(selected_column).write_csv(f"{FULL_PREFIX}{name}.csv")

def count_fams(d): return d[HOMOLOGY_COL].n_unique()

print(f"{'TRAIN':<10} | {len(train_df):>8} | {count_fams(train_df):>8} | {len(train_df) / total_rows * 100:>5.1f}%")
print(f"{'VAL':<10} | {len(val_df):>8} | {count_fams(val_df):>8} | {len(val_df) / total_rows * 100:>5.1f}%")
print(f"{'TEST':<10} | {len(test_df):>8} | {count_fams(test_df):>8} | {len(test_df) / total_rows * 100:>5.1f}%")



save(train_df, "train")
save(val_df, "validation")
save(test_df, "test")

if "reverse" in df.columns:
    save(test_df.filter(pl.col("reverse") == False), "test_noreverse")

print(f"Hotovo. Data s PLNOU délkou uložena do {BASE_DIR}")

Zahajuji deterministický split (zachování celých sekvencí)...
TRAIN      |  1361598 |      139 |  78.3%
VAL        |   188631 |       45 |  10.8%
TEST       |   188632 |       45 |  10.8%
Hotovo. Data s PLNOU délkou uložena do datasets


In [5]:
train_df

original_seq_full,mutated_seq_full,mut_type,target,reverse,data_source,cath_dominant,cath_all,cath_class,cath_arch,cath_topology,cath_homology
str,str,str,f64,bool,str,str,str,str,str,str,str
"""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""SAGGSAGGSAGGSNSLAEAKVLANRELDKY…","""I27T""",-0.352598,false,"""megascale""","""G3DSA:1.20.900.10""","""G3DSA:1.10.238.10;G3DSA:1.20.9…","""G3DSA:1""","""20""","""900""","""10"""
"""SAGGSEVTIKANLIFANGSTQTAEFKGTFE…","""SAGGSEVTIKANLIFANGSGQTAEFKGTFE…","""T15G""",-0.166278,false,"""megascale""","""G3DSA:1.10.10.10""","""G3DSA:1.10.10.10;G3DSA:1.10.15…","""G3DSA:1""","""10""","""10""","""10"""
"""SAGGSAGGTFTSIVTTNPDFGGFEFYVEAG…","""SAGGSAGGTKTSIVTTNPDFGGFEFYVEAG…","""F2K:H63L""",-0.251796,false,"""megascale""","""G3DSA:3.30.1370.10""","""G3DSA:3.30.1370.10""","""G3DSA:3""","""30""","""1370""","""10"""
"""SAGGSATGEEYIAVGDFTAQQVGDLTFKKG…","""SAGGSATGEEYIAVGDFTAQQVGDLTFKRG…","""K23R""",0.020005,false,"""megascale""","""G3DSA:6.10.280.50""","""G3DSA:6.10.280.50""","""G3DSA:6""","""10""","""280""","""50"""
"""SAGGSAGGSAGGVKKMAKAIMADPNKADEV…","""SAGGSAGGSAGGVKKMAKAIMADPNKADEV…","""T40P""",0.10965,false,"""megascale""","""G3DSA:4.10.320.10""","""G3DSA:4.10.320.10""","""G3DSA:4""","""10""","""320""","""10"""
…,…,…,…,…,…,…,…,…,…,…,…
"""MSNSRNNRVMVEGVGARVVRGPDWKWGKQD…","""MSNSRNNRVMVEGVGARVVRGPDWKWGKQD…","""S247Q""",0.027341,false,"""lehner""","""G3DSA:2.30.42.10""","""G3DSA:2.20.70.10;G3DSA:2.30.42…","""G3DSA:2""","""30""","""42""","""10"""
"""MSNSRNNRVMVEGVGARVVRGPDWKWGKQD…","""MSNSRNNRVMVEGVGARVVRGPDWKWGKQD…","""T247Q""",-0.018051,false,"""lehner""","""G3DSA:2.30.42.10""","""G3DSA:2.20.70.10;G3DSA:2.30.42…","""G3DSA:2""","""30""","""42""","""10"""
"""MSNSRNNRVMVEGVGARVVRGPDWKWGKQD…","""MSNSRNNRVMVEGVGARVVRGPDWKWGKQD…","""V247Q""",-0.050482,false,"""lehner""","""G3DSA:2.30.42.10""","""G3DSA:2.20.70.10;G3DSA:2.30.42…","""G3DSA:2""","""30""","""42""","""10"""


In [6]:
df_255.filter(pl.col("original_seq_full").is_in(holdout_sequences))

NameError: name 'holdout_sequences' is not defined

In [ ]:
holdout_sequences